In [ ]:
import numpy as np

class AnomalyDetector:
    def __init__(self, threshold=7):
        self.threshold = threshold
        self.mean = np.array([0.0, 0.0, 0.0, 0.0])
        self.var = np.array([0.0, 0.0, 0.0, 0.0])
        self.count = 0
        self.prev_data = None

    def update_statistics(self, data):
        self.count += 1
        if self.count == 1:
            self.mean = data
            self.var = np.array([0.0, 0.0, 0.0, 0.0])
        else:
            new_mean = self.mean + (data - self.mean) / self.count
            self.var = (1 - 1/(self.count - 1)) * self.var + (data - self.mean) * (data - new_mean)
            self.mean = new_mean
    
    def calculate_z_scores(self, data):
        if self.count <= 1:
            return np.zeros(4)
        std_dev = np.sqrt(self.var / (self.count - 1))
        z_scores = (data - self.mean) / std_dev
        return z_scores
    
    def detect_anomaly(self, data):
        if self.prev_data is not None:
            diff_data = data - self.prev_data
            self.update_statistics(diff_data)
            z_scores = self.calculate_z_scores(diff_data)
            is_anomalous = any(abs(z) > self.threshold for z in z_scores)
        else:
            is_anomalous = False
        self.prev_data = data
        return is_anomalous

# Example usage
anomaly_detector = AnomalyDetector()

data_stream = [
    (100.0, 105.0, 95.0, 102.0),
    (101.0, 104.0, 96.0, 103.0),
    (99.0, 106.0, 94.0, 104.0),
    (98.0, 105.0, 96.0, 100.0),
    (105.0, 107.0, 97.0, 101.0), # This should be flagged as anomaly
    # Add more tuples of (open, high, low, close) data
]

for data in data_stream:
    if anomaly_detector.detect_anomaly(np.array(data)):
        print(f"Anomaly detected in data: {data}")
